# MusicRecoSystem — Artist Recommendation with K-Means

## 1. Introduction

### Project objective

The objective of this notebook is to build an artist recommendation
system using K-Means clustering.

The model groups users according to their listening habits and
recommends artists that are popular among users in the same cluster.

### Methodology

1. Load and prepare listening data.
2. Split the dataset into training, validation and test sets.
3. Construct a user-artist interaction matrix.
4. Normalize user listening profiles.
5. Apply K-Means clustering.
6. Generate artist recommendations.
7. Evaluate recommendation quality using Recall@10 and NDCG@10.
8. Optimize the number of clusters
9. final evaluation
10. Conclusion

### Evaluation strategy

The model is trained using the training set.

The validation set is used to select the optimal number of clusters.

The test set is reserved for the final comparison with other
recommendation models.

In [405]:
import pandas as pd
import numpy as np

from pathlib import Path

# 1 LOAD DATA

In [406]:

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"

df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

(9974, 6)


,user,artist,artist_mbids,track,release,listened_at
0,jaymiles,Billie Eilish,[f4abc0b5-3f7a-4eff-8f78-ac078dbce533],CHIHIRO,HIT ME HARD AND SOFT,2026-10-12 12:14:04+00:00
1,jaymiles,Billie Eilish,[f4abc0b5-3f7a-4eff-8f78-ac078dbce533],CHIHIRO,HIT ME HARD AND SOFT,2026-10-12 11:58:03+00:00
2,jaymiles,Playboi Carti,[2baf3276-ed6a-4349-8d2e-f4601e7b2167],Yah Mean,Playboi Carti,2026-10-12 11:54:58+00:00
3,jaymiles,Playboi Carti,"[2baf3276-ed6a-4349-8d2e-f4601e7b2167, 800760d...",Choppa Won't Miss (feat. Young Thug),Die Lit,2026-10-12 11:51:15+00:00
4,jaymiles,Grimes,[7e5a2a59-6d9f-4a17-b7c2-e1eedb7bd222],Genesis,Visions,2026-10-12 11:46:38+00:00


In [407]:
print(df.shape)
print(df.columns.tolist())

(9974, 6)
['user', 'artist', 'artist_mbids', 'track', 'release', 'listened_at']


## 2 Data Preparation

### Chronological ordering

Listening events are sorted chronologically for each user.

This ensures that the recommendation models learn from past
listening behavior and are evaluated on subsequent interactions.

In [408]:
df = df.sort_values(
    by=["user", "listened_at"]
).reset_index(drop=True)

df[["user", "artist", "listened_at"]].head(10)

,user,artist,listened_at
0,AsuraDtr,Moji x Sboy,2026-04-04 19:30:07+00:00
1,AsuraDtr,Ino Casablanca,2026-04-04 19:32:26+00:00
2,AsuraDtr,Damso,2026-04-04 19:36:12+00:00
3,AsuraDtr,"ISHA, Limsa d'Aulnay",2026-04-04 19:38:51+00:00
4,AsuraDtr,"Luidji, Ryan Koffi",2026-04-05 10:21:57+00:00
5,AsuraDtr,Bekar,2026-04-05 12:31:08+00:00
6,AsuraDtr,BEN plg,2026-04-05 12:35:39+00:00
7,AsuraDtr,disiz,2026-04-05 12:38:00+00:00
8,AsuraDtr,"Ino Casablanca, Nensi",2026-04-05 12:40:35+00:00
9,AsuraDtr,TIF,2026-04-05 12:44:02+00:00


### Train / Validation / Test Split

We split each user's listening history chronologically:

- 80% for training
- 10% for validation
- 10% for testing

This approach allows us to evaluate recommendations on future
listening events rather than randomly selected past interactions.

In [409]:
train_parts = []
val_parts = []
test_parts = []

for user, group in df.groupby("user"):

    n = len(group)

    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_parts.append(group.iloc[:train_end])
    val_parts.append(group.iloc[train_end:val_end])
    test_parts.append(group.iloc[val_end:])

# 3. User-Artist Interaction Matrix

###  Counting User-Artist Interactions

We aggregate the training data to count how many times each
user listened to each artist.

These interaction counts will be used to build a numerical
representation of users' listening preferences.

In [410]:
train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print("Train :", train_df.shape)
print("Validation :", val_df.shape)
print("Test :", test_df.shape)

Train : (7979, 6)
Validation : (997, 6)
Test : (998, 6)


In [411]:
train_interactions = (
    train_df
    .groupby(["user", "artist"])
    .size()
    .reset_index(name="count")
)

print(train_interactions.shape)

train_interactions.head(10)

(4004, 3)


,user,artist,count
0,AsuraDtr,BEN plg,5
1,AsuraDtr,"BEN plg, COLORS",1
2,AsuraDtr,Bekar,5
3,AsuraDtr,Bigflo & Oli,2
4,AsuraDtr,Damso,3
5,AsuraDtr,Danyl,7
6,AsuraDtr,"Danyl, Stony Stone",2
7,AsuraDtr,Dinos,1
8,AsuraDtr,Fleetzy,4
9,AsuraDtr,Folie's,1


### User-Artist Matrix

We transform user-artist interaction counts into a matrix where:

- Rows represent users.
- Columns represent artists.
- Values represent listening frequencies.

Missing interactions are replaced with zero.

In [412]:
user_artist_matrix = train_interactions.pivot(
    index="user",
    columns="artist",
    values="count"
).fillna(0)

print(user_artist_matrix.shape)

user_artist_matrix.iloc[:5, :10]

(100, 2639)


artist,$NOT,"$NOT, A$AP Rocky",*67,-M-,.Feast,070 Shake,10cc,13 Block,1400 Vault,1PLIKATION
user,,,,,,,,,,
AsuraDtr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BKILPATRICK11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Evan music,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Giganigga69,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gonca12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 4. L2 Normalization

User listening frequencies can vary significantly.

To reduce the influence of listening volume, we apply L2
normalization to each user vector.

This transformation preserves relative listening preferences
while ensuring that each non-zero user vector has a norm of 1.

In [413]:
from sklearn.preprocessing import normalize

normalized_matrix = normalize(
    user_artist_matrix,
    norm="l2"
)

print(normalized_matrix.shape)

(100, 2639)


## 5. K-Means Clustering

###  Model Training

K-Means is an unsupervised learning algorithm that groups
users according to similarities in their listening profiles.

The model is trained using the L2-normalized user-artist matrix.

We initially use K=3, which was identified as the best
configuration during our previous validation experiments.

In [414]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(normalized_matrix)

print(cluster_labels[:10])

[0 1 1 0 1 2 1 0 0 1]


In [415]:
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()

print(cluster_counts)

0    42
1    38
2    20
Name: count, dtype: int64


### User Cluster Assignment

Each user is assigned to one of the three clusters.

We create a DataFrame mapping each user to their assigned
cluster to analyze listening preferences within each group.

In [416]:
user_clusters = pd.DataFrame({
    "user": user_artist_matrix.index,
    "cluster": cluster_labels
})

user_clusters.head(10)

,user,cluster
0,AsuraDtr,0
1,BKILPATRICK11,1
2,Evan music,1
3,Giganigga69,0
4,Gonca12,1
5,HHTHH,2
6,InnerOtaku,1
7,Kikay,0
8,Kilo.cozel52,0
9,Kraisky,1


### Cluster Listening Analysis

We combine user-artist interactions with cluster assignments
to identify the listening preferences of each cluster.

This allows us to analyze which artists are popular among
users belonging to the same group.

In [417]:
cluster_interactions = train_interactions.merge(
    user_clusters,
    on="user",
    how="left",
    validate="many_to_one"
)

print(cluster_interactions.shape)

cluster_interactions.head(10)

(4004, 4)


,user,artist,count,cluster
0,AsuraDtr,BEN plg,5,0
1,AsuraDtr,"BEN plg, COLORS",1,0
2,AsuraDtr,Bekar,5,0
3,AsuraDtr,Bigflo & Oli,2,0
4,AsuraDtr,Damso,3,0
5,AsuraDtr,Danyl,7,0
6,AsuraDtr,"Danyl, Stony Stone",2,0
7,AsuraDtr,Dinos,1,0
8,AsuraDtr,Fleetzy,4,0
9,AsuraDtr,Folie's,1,0


###  Artist Popularity Within Clusters

We measure artist popularity using the number of distinct
users who listened to each artist within a cluster.

This approach reduces the influence of individual users
who repeatedly listen to the same artist.

In [418]:
cluster_artist_users = (
    cluster_interactions
    .groupby(["cluster", "artist"])["user"]
    .nunique()
    .reset_index(name="n_users")
)

print(cluster_artist_users.shape)

cluster_artist_users.head(10)

(2971, 3)


,cluster,artist,n_users
0,0,*67,1
1,0,-M-,1
2,0,10cc,1
3,0,13 Block,1
4,0,1PLIKATION,1
5,0,1PLIKÉ140,1
6,0,1oneam,1
7,0,2 Mello,1
8,0,2Pac,1
9,0,"2Pac, Roger, Dr. Dre",1


###  Ranking Artists by Cluster Popularity

Artists are ranked within each cluster according to the
number of distinct users who listened to them.

The most popular artists will be considered first when
generating recommendations.

In [419]:
for cluster in sorted(cluster_artist_users["cluster"].unique()):

    print(f"\nCluster {cluster}")

    display(
        cluster_artist_users[
            cluster_artist_users["cluster"] == cluster
        ].head(10)
    )


Cluster 0


,cluster,artist,n_users
0,0,*67,1
1,0,-M-,1
2,0,10cc,1
3,0,13 Block,1
4,0,1PLIKATION,1
5,0,1PLIKÉ140,1
6,0,1oneam,1
7,0,2 Mello,1
8,0,2Pac,1
9,0,"2Pac, Roger, Dr. Dre",1



Cluster 1


,cluster,artist,n_users
1365,1,$NOT,2
1366,1,"$NOT, A$AP Rocky",1
1367,1,.Feast,1
1368,1,070 Shake,1
1369,1,1400 Vault,1
1370,1,21 Savage,11
1371,1,21 Savage & Drake,1
1372,1,21 Savage & G Herbo,1
1373,1,21 Savage & Young Nudy,1
1374,1,"21 Savage, Drake",1



Cluster 2


,cluster,artist,n_users
2336,2,"21 Savage, Travis Scott, Metro Boomin",1
2337,2,"50 Cent, The Madd Rapper",1
2338,2,88rising & NIKI,1
2339,2,905aido,1
2340,2,A Boogie Wit da Hoodie,1
2341,2,A Boogie wit da Hoodie,1
2342,2,A$AP Ferg,1
2343,2,A$AP Rocky,6
2344,2,"A$AP Rocky, Doechii",1
2345,2,"A$AP Rocky, Drake, 2 Chainz, Kendrick Lamar",1


# 6. Artist Recommendation System

We generate artist recommendations based on the user's
assigned cluster.

The system recommends popular artists within the cluster
while excluding artists already present in the user's
training listening history.

### User-to-Cluster Mapping

In [420]:
user_to_cluster = dict(
    zip(user_clusters["user"], user_clusters["cluster"])
)

list(user_to_cluster.items())[:5]

[('AsuraDtr', 0),
 ('BKILPATRICK11', 1),
 ('Evan music', 1),
 ('Giganigga69', 0),
 ('Gonca12', 1)]

### User Listening History

We identify the artists each user has already listened to
in the training dataset.

These artists will be excluded from future recommendations
to ensure that the system suggests new artists.

In [421]:
train_artists = (
    train_df.groupby("user")["artist"]
    .apply(set)
    .to_dict()
)

list(train_artists.items())[:2]

[('AsuraDtr',
  {'BEN plg',
   'BEN plg, COLORS',
   'Bekar',
   'Bigflo & Oli',
   'Damso',
   'Danyl',
   'Danyl, Stony Stone',
   'Dinos',
   'Fleetzy',
   "Folie's",
   'GP Explorer, Charlotte Cardin, SCH',
   'Hamza',
   "ISHA, Limsa d'Aulnay",
   'Ino Casablanca',
   'Ino Casablanca, Nensi',
   'Jody',
   'Luidji, Ryan Koffi',
   'Moji x Sboy',
   "Oxmo Puccino, Danyl, Jok'air",
   'PLK, Hamza',
   'PLK, OBOY',
   'ROBB',
   'ROBB, Mick',
   'Rounhaa',
   'TIF',
   'Theodora',
   'Trinix',
   'Trinix, Bonga',
   'Wamen',
   'Yamê',
   'aupinard',
   'disiz',
   'disiz, Iliona'}),
 ('BKILPATRICK11',
  {'A$AP Rocky',
   'Anderson .Paak',
   'Baby Keem',
   'Blur',
   'Busta Rhymes',
   'Clipse',
   'Dahi',
   'Danger Mouse',
   'Danny Brown',
   'Denzel Curry',
   'Erykah Badu',
   'Gorillaz',
   'J. Cole',
   'JAŸ-Z',
   'JID',
   'JPEGMAFIA',
   'Joy Crookes',
   'Kanye West',
   'Kendrick Lamar',
   'Kid Cudi',
   'Little Simz',
   'Logic',
   'Lupe Fiasco',
   'Mac Miller',
   

In [422]:
cluster_artist_users = cluster_artist_users.sort_values(
    by=["cluster", "n_users"],
    ascending=[True, False]
)

cluster_artist_users.groupby("cluster").head(5)

,cluster,artist,n_users
809,0,Ninho,8
926,0,Radiohead,8
918,0,ROSALÍA,7
249,0,Damso,6
452,0,Hamza,6
2188,1,Travis Scott,22
1798,1,Kanye West,20
1872,1,Lil Uzi Vert,20
1588,1,Don Toliver,19
1602,1,Drake,19


###  Cluster Artist Rankings

We create a dictionary containing artists ranked by
popularity within each cluster.

These rankings will be used to generate personalized
artist recommendations.

In [423]:
cluster_top_artists = (
    cluster_artist_users
    .groupby("cluster")["artist"]
    .apply(list)
    .to_dict()
)

cluster_top_artists[0][:10]

['Ninho',
 'Radiohead',
 'ROSALÍA',
 'Damso',
 'Hamza',
 'Josman',
 'Arctic Monkeys',
 'Booba',
 'Drake',
 'Gorillaz']

### K-Means Recommendation Function

The recommendation function identifies the user's cluster
and retrieves the most popular artists within that cluster.

Artists already present in the user's training history
are excluded to encourage music discovery.

In [424]:
def recommend_kmeans(user, top_n=10):

    cluster = user_to_cluster[user]

    known_artists = train_artists[user]

    popular_artists = cluster_top_artists[cluster]

    recommendations = [
        artist
        for artist in popular_artists
        if artist not in known_artists
    ]

    return recommendations[:top_n]

### TEST

In [425]:
recommend_kmeans("BKILPATRICK11")

['Travis Scott',
 'Lil Uzi Vert',
 'Don Toliver',
 'Drake',
 'Playboi Carti',
 'Future',
 'Gunna',
 'Yeat',
 '21 Savage',
 'The Weeknd']

## 6. Model Evaluation

We evaluate the recommendation system using validation data.

Two ranking metrics are considered:
- Recall@10
- NDCG@10

### Relevant Artists

Relevant artists are those appearing in a user's validation
history but not in their training history.

Only artists available in the training catalog are considered
evaluable.

In [426]:
train_catalog = set(train_df["artist"])

val_artists = (
    val_df.groupby("user")["artist"]
    .apply(set)
)

#
evaluable_val_artists = val_artists.apply(
    lambda artists: artists & train_catalog
)

evaluable_val_artists.head()

user
AsuraDtr         {Laylow, TIF, Nono La Grinta, Hamza, RnBoi, Fo...
BKILPATRICK11    {Nas, Declan McKenna, Baby Keem, Steve Lacy, J...
Evan music                                                 {Damso}
Giganigga69      {Jackson Laird, Mac DeMarco, Malcolm Todd, Rad...
Gonca12          {LON3R JOHNY, Mizzy Miles, T-Rex, Plutonio, Ju...
Name: artist, dtype: object

In [427]:
evaluable_val_artists = evaluable_val_artists.apply(
    lambda artists: artists
)

for user in evaluable_val_artists.index:
    evaluable_val_artists[user] = (
        evaluable_val_artists[user] - train_artists[user]
    )

evaluable_val_artists.head()

user
AsuraDtr                {Nono La Grinta, Laylow, RnBoi, Josman}
BKILPATRICK11    {Joey Bada$$, Steve Lacy, Nas, Declan McKenna}
Evan music                                                   {}
Giganigga69             {Jackson Laird, Coldplay, Malcolm Todd}
Gonca12                                                      {}
Name: artist, dtype: object

### Evaluable Users

Only users with at least one relevant artist in the
validation dataset are included in the evaluation.

This ensures that Recall@10 can be calculated.

In [428]:
evaluable_val_artists = evaluable_val_artists[
    evaluable_val_artists.apply(len) > 0
]

print("Evaluable users:", len(evaluable_val_artists))

Evaluable users: 69


### Recall@10

Recall@10 measures the proportion of relevant artists
retrieved among the top 10 recommendations.

A higher Recall@10 indicates that the model identifies
more artists that users listen to during validation.

In [429]:
def recall_at_k(recommendations, relevant_artists):

    recommended = set(recommendations)

    hits = recommended & relevant_artists

    return len(hits) / len(relevant_artists)

### Average Recall@10

We evaluate the K-Means recommendation model across
all evaluable users.

For each user, we generate 10 artist recommendations
and calculate Recall@10.

The final score is the average Recall@10 across users.

In [430]:
recall_scores = []

for user, relevant_artists in evaluable_val_artists.items():

    recommendations = recommend_kmeans(user, top_n=10)

    score = recall_at_k(recommendations, relevant_artists)

    recall_scores.append(score)

mean_recall = np.mean(recall_scores)

print("K-Means Recall@10:", mean_recall)

K-Means Recall@10: 0.15386473429951691


### NDCG@10

Normalized Discounted Cumulative Gain (NDCG@10) evaluates
the quality of recommendation rankings.

Unlike Recall@10, NDCG@10 gives more importance to
relevant artists appearing at higher positions.

In [431]:
def ndcg_at_k(recommendations, relevant_artists):

    dcg = 0

    for i, artist in enumerate(recommendations):

        if artist in relevant_artists:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant_artists), len(recommendations))

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0

###  Average NDCG@10

We calculate the average NDCG@10 across all evaluable users.

This metric measures how effectively the recommendation
system ranks relevant artists near the top of the list.

In [432]:
ndcg_scores = []

for user, relevant_artists in evaluable_val_artists.items():

    recommendations = recommend_kmeans(user, top_n=10)

    score = ndcg_at_k(recommendations, relevant_artists)

    ndcg_scores.append(score)

mean_ndcg = np.mean(ndcg_scores)

print("K-Means NDCG@10:", mean_ndcg)

K-Means NDCG@10: 0.10246402263437095


# 8. K-Means Hyperparameter Tuning

We evaluate different numbers of clusters to identify
the configuration that produces the best recommendations.

The number of clusters (k) is selected using validation
performance, measured by Recall@10 and NDCG@10.

### Training Multiple K-Means Models

In [433]:
k_values = [2, 3, 4, 5, 6, 8, 10, 12, 13,15, 20]

kmeans_models = {}

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(normalized_matrix)

    kmeans_models[k] = labels

    print(f"k={k}: {len(set(labels))} clusters")

k=2: 2 clusters
k=3: 3 clusters
k=4: 4 clusters
k=5: 5 clusters
k=6: 6 clusters
k=8: 8 clusters
k=10: 10 clusters
k=12: 12 clusters
k=13: 13 clusters
k=15: 15 clusters
k=20: 20 clusters


### Building Cluster-Specific Artist Rankings

For each K-Means configuration, we assign users to clusters
and calculate artist popularity within each cluster.

Artist popularity is measured by the number of distinct
users who listened to each artist.

###  Automated Artist Rankings

The function builds artist rankings for any K-Means
configuration.

Artists are ranked according to the number of distinct
listeners within each cluster.

In [434]:
def build_cluster_rankings(labels):

    clusters = pd.DataFrame({
        "user": user_artist_matrix.index,
        "cluster": labels
    })

    interactions = train_interactions.merge(
        clusters,
        on="user",
        how="left",
        validate="many_to_one"
    )

    artist_popularity = (
        interactions
        .groupby(["cluster", "artist"])["user"]
        .nunique()
        .reset_index(name="n_users")
    )

    artist_popularity = artist_popularity.sort_values(
        by=["cluster", "n_users"],
        ascending=[True, False]
    )

    rankings = (
        artist_popularity
        .groupby("cluster")["artist"]
        .apply(list)
        .to_dict()
    )

    return rankings

### Recommendation Function for Different K Values

We create a recommendation function that works with any
K-Means configuration.

The function uses cluster assignments and artist rankings
to recommend artists not present in the user's training history.

In [435]:
def recommend_with_clusters(user, labels, rankings, top_n=10):

    user_index = user_artist_matrix.index.get_loc(user)

    cluster = labels[user_index]

    known_artists = train_artists[user]

    popular_artists = rankings[cluster]

    recommendations = [
        artist
        for artist in popular_artists
        if artist not in known_artists
    ]

    return recommendations[:top_n]

In [436]:
rankings_k3 = build_cluster_rankings(kmeans_models[3])
old_recommendations = recommend_kmeans("BKILPATRICK11")

new_recommendations = recommend_with_clusters(
    "BKILPATRICK11",
    kmeans_models[3],
    rankings_k3
)

print("Old:", old_recommendations)
print("New:", new_recommendations)

print("Identical:", old_recommendations == new_recommendations)

Old: ['Travis Scott', 'Lil Uzi Vert', 'Don Toliver', 'Drake', 'Playboi Carti', 'Future', 'Gunna', 'Yeat', '21 Savage', 'The Weeknd']
New: ['Travis Scott', 'Lil Uzi Vert', 'Don Toliver', 'Drake', 'Playboi Carti', 'Future', 'Gunna', 'Yeat', '21 Savage', 'The Weeknd']
Identical: True


### Evaluating K-Means Configurations

We evaluate each K-Means configuration using Recall@10
and NDCG@10 on the validation dataset.

The evaluation uses the same users and relevant artists
for every configuration to ensure a fair comparison.

In [437]:
def evaluate_kmeans(labels, rankings):

    recall_scores = []
    ndcg_scores = []

    for user, relevant_artists in evaluable_val_artists.items():

        recommendations = recommend_with_clusters(
            user, labels, rankings, top_n=10
        )

        recall_scores.append(
            recall_at_k(recommendations, relevant_artists)
        )

        ndcg_scores.append(
            ndcg_at_k(recommendations, relevant_artists)
        )

    return np.mean(recall_scores), np.mean(ndcg_scores)

###  Hyperparameter Comparison

We compare different numbers of clusters using
Recall@10 and NDCG@10.

The objective is to identify the K-Means configuration
that produces the best recommendations on validation data.

In [438]:
results = []

for k in k_values:

    labels = kmeans_models[k]

    rankings = build_cluster_rankings(labels)

    recall, ndcg = evaluate_kmeans(labels, rankings)

    results.append({
        "k": k,
        "recall@10": recall,
        "ndcg@10": ndcg
    })

results_df = pd.DataFrame(results)

results_df

,k,recall@10,ndcg@10
0,2,0.165942,0.124018
1,3,0.153865,0.102464
2,4,0.156280,0.121209
3,5,0.171981,0.129167
4,6,0.187681,0.117532
5,8,0.153382,0.097567
6,10,0.100725,0.087935
7,12,0.160628,0.107227
8,13,0.138889,0.099246
9,15,0.164251,0.123322


### Best K-Means Configuration

The optimal number of clusters is selected based on
validation NDCG@10.

The best configuration uses k=5, achieving:

- Recall@10: 0.1719
- NDCG@10: 0.1291

The selected configuration will be used for final
evaluation on the held-out test dataset.

In [439]:
best_row = results_df.loc[results_df["ndcg@10"].idxmax()]

best_k = int(best_row["k"])

print("Best k:", best_k)
print("Recall@10:", best_row["recall@10"])
print("NDCG@10:", best_row["ndcg@10"])

Best k: 5
Recall@10: 0.17198067632850242
NDCG@10: 0.12916746921634203


# 9 Preparing the Final Training Dataset

After selecting the optimal number of clusters using
validation data, we combine the training and validation
datasets.

The test dataset remains untouched and will only be
used for final evaluation.

In [440]:
final_train_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

print("Final training:", final_train_df.shape)
print("Test:", test_df.shape)

Final training: (8976, 6)
Test: (998, 6)


###  Building the Final User-Artist Matrix

We rebuild the user-artist interaction matrix using
the combined training and validation datasets.

Each interaction represents the number of times
a user listened to an artist.

In [441]:
final_interactions = (
    final_train_df
    .groupby(["user", "artist"])
    .size()
    .reset_index(name="listen_count")
)

final_interactions.head()

,user,artist,listen_count
0,AsuraDtr,BEN plg,5
1,AsuraDtr,"BEN plg, COLORS",1
2,AsuraDtr,Bekar,5
3,AsuraDtr,Bigflo & Oli,2
4,AsuraDtr,Damso,3


### Final User-Artist Matrix and Normalization

We convert the final user-artist interactions into a matrix.

Each row represents a user, and each column represents an artist.

We apply L2 normalization to focus on relative listening
preferences rather than total listening volume.

In [442]:
final_user_artist_matrix = final_interactions.pivot(
    index="user",
    columns="artist",
    values="listen_count"
).fillna(0)

print(final_user_artist_matrix.shape)

(100, 2878)


### Training the Final K-Means Model

We train the final K-Means model using the selected
number of clusters (k=3).

The model is fitted on the combined training and
validation data, while the test set remains untouched.

In [443]:
from sklearn.preprocessing import normalize

final_normalized_matrix = normalize(
    final_user_artist_matrix,
    norm="l2"
)

print(final_normalized_matrix.shape)

(100, 2878)


In [444]:
final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

final_labels = final_kmeans.fit_predict(
    final_normalized_matrix
)

print("Number of clusters:", len(set(final_labels)))
print("Cluster sizes:")

print(pd.Series(final_labels).value_counts())

Number of clusters: 5
Cluster sizes:
1    39
4    30
3    11
2    11
0     9
Name: count, dtype: int64


### Building Final Cluster Artist Rankings

We rebuild artist popularity rankings using the final
training interactions and the updated cluster assignments.

Artist popularity is measured by the number of distinct
users who listened to each artist within a cluster.

In [445]:
def build_final_cluster_rankings(labels, interactions, users):

    clusters = pd.DataFrame({
        "user": users,
        "cluster": labels
    })

    cluster_interactions = interactions.merge(
        clusters,
        on="user",
        how="left",
        validate="many_to_one"
    )

    artist_popularity = (
        cluster_interactions
        .groupby(["cluster", "artist"])["user"]
        .nunique()
        .reset_index(name="n_users")
    )

    artist_popularity = artist_popularity.sort_values(
        by=["cluster", "n_users"],
        ascending=[True, False]
    )

    rankings = (
        artist_popularity
        .groupby("cluster")["artist"]
        .apply(list)
        .to_dict()
    )

    return rankings

In [446]:
final_rankings = build_final_cluster_rankings(
    final_labels,
    final_interactions,
    final_user_artist_matrix.index
)

for cluster, artists in final_rankings.items():
    print(f"Cluster {cluster}: {artists[:5]}")

Cluster 0: ['ROSALÍA', 'ROSALÍA, Björk, Yves Tumor', 'ROSALÍA, Carminho', 'ROSALÍA, Dougie F', 'ROSALÍA, Estrella Morente, Sílvia Pérez Cruz']
Cluster 1: ['Don Toliver', 'Drake', 'Playboi Carti', 'Future', 'Lil Uzi Vert']
Cluster 2: ['Malcolm Todd', 'Mac DeMarco', 'Steve Lacy', 'Daniel Caesar', "Her's"]
Cluster 3: ['Ninho', 'Damso', 'Hamza', 'Josman', 'Booba']
Cluster 4: ['Frank Ocean', 'Kanye West', 'Kendrick Lamar', 'Steve Lacy', 'Tyler, The Creator']


### Preparing Final Recommendations

We identify the artists already listened to by each user
in the combined training and validation datasets.

These artists will be excluded from recommendations
to focus on new artist discovery.

In [447]:
final_train_artists = (
    final_train_df
    .groupby("user")["artist"]
    .apply(set)
    .to_dict()
)

In [448]:
def recommend_final(user, top_n=10):

    user_index = final_user_artist_matrix.index.get_loc(user)

    cluster = final_labels[user_index]

    known_artists = final_train_artists[user]

    popular_artists = final_rankings[cluster]

    recommendations = [
        artist
        for artist in popular_artists
        if artist not in known_artists
    ]

    return recommendations[:top_n]

In [449]:
recommend_final("BKILPATRICK11")

['Frank Ocean',
 'Daniel Caesar',
 'The Weeknd',
 'Drake',
 'Travis Scott',
 'Don Toliver',
 'PinkPantheress',
 'Playboi Carti',
 'SZA',
 'Brent Faiyaz']

### Preparing Test Evaluation

We identify relevant artists from the held-out test dataset.

An artist is considered relevant if:
- The user listened to the artist during the test period.
- The artist was not previously listened to during training.
- The artist exists in the training catalog.

This ensures that recommendations are evaluated on
new artists that the model could potentially recommend.

In [450]:
final_train_catalog = set(final_train_df["artist"])

test_artists = (
    test_df
    .groupby("user")["artist"]
    .apply(set)
)

evaluable_test_artists = test_artists.copy()

for user in evaluable_test_artists.index:

    relevant = test_artists[user] & final_train_catalog

    relevant = relevant - final_train_artists[user]

    evaluable_test_artists[user] = relevant

evaluable_test_artists = evaluable_test_artists[
    evaluable_test_artists.apply(len) > 0
]

print("Evaluable test users:", len(evaluable_test_artists))

Evaluable test users: 71


###  Final K-Means Test Evaluation

We evaluate the final K-Means recommender on the held-out
test dataset using Recall@10 and NDCG@10.

The model was trained on the combined training and
validation datasets.

Test data is used only for final performance evaluation.

In [451]:
test_recalls = []
test_ndcgs = []

for user, relevant_artists in evaluable_test_artists.items():

    recommendations = recommend_final(user, top_n=10)

    test_recalls.append(
        recall_at_k(recommendations, relevant_artists)
    )

    test_ndcgs.append(
        ndcg_at_k(recommendations, relevant_artists)
    )

final_recall = np.mean(test_recalls)
final_ndcg = np.mean(test_ndcgs)

print("Final Recall@10:", final_recall)
print("Final NDCG@10:", final_ndcg)

Final Recall@10: 0.17981220657276997
Final NDCG@10: 0.11818401673528381


###  Final Comparison with Popularity Baseline

We evaluate a global popularity recommender on the
same held-out test dataset as the final K-Means model.

This comparison helps determine whether clustering
improves recommendation quality over a simple baseline.

In [452]:
global_popularity = (
    final_interactions
    .groupby("artist")["user"]
    .nunique()
    .sort_values(ascending=False)
)

popular_artists = global_popularity.index.tolist()

### Evaluating the Popularity Baseline

We evaluate the global popularity recommender on the
held-out test dataset.

Previously listened artists are excluded to focus
on new artist discovery.

In [453]:
def recommend_popularity_final(user, top_n=10):

    known_artists = final_train_artists[user]

    recommendations = [
        artist
        for artist in popular_artists
        if artist not in known_artists
    ]

    return recommendations[:top_n]

In [454]:
pop_recalls = []
pop_ndcgs = []

for user, relevant_artists in evaluable_test_artists.items():

    recommendations = recommend_popularity_final(user, top_n=10)

    pop_recalls.append(
        recall_at_k(recommendations, relevant_artists)
    )

    pop_ndcgs.append(
        ndcg_at_k(recommendations, relevant_artists)
    )

pop_recall = np.mean(pop_recalls)
pop_ndcg = np.mean(pop_ndcgs)

print("Popularity Recall@10:", pop_recall)
print("Popularity NDCG@10:", pop_ndcg)

Popularity Recall@10: 0.1461658841940532
Popularity NDCG@10: 0.10175223505713929


# 10 Conclusion

We developed and evaluated a music recommendation system
based on K-Means clustering.

The optimal configuration used 5 clusters, selected
using validation NDCG@10.

Final test results:

| Model | Recall@10 | NDCG@10 |
|---|---|---|
| Global Popularity | 0.14616 | 0.10175 |
| K-Means (k=5) | 0.1719 | 0.1182 |



These results suggest that clustering users based
on listening preferences can improve recommendation
quality.

However, performance remains limited because
recommendations are based on cluster-level popularity
rather than individual user similarities.

Future experiments will explore nearest-neighbor
collaborative filtering, XGBoost, and neural networks.